# MLP and Transformer Split-Horizon Forecasting

This notebook uses one shared BasicTS forecasting data pipeline for both models. The MLP forecasts steps 1-6, the Transformer forecasts steps 7-12, and their 6-step outputs are concatenated into one 12-step hybrid forecast.


## 1. Project Setup

This cell keeps the notebook runnable from inside `notebooks/` by moving to the repo root and adding `src/` to Python's import path.


In [13]:
import os 
import sys
from pathlib import Path
#root contains the path to the basicts
ROOT = Path(r"C:\Users\luwil\OneDrive\Documents\Code\BasicTS")
#move python working folder
os.chdir(ROOT)
# the path to src is src_path
src_path = ROOT / "src"
#if src_path is not in the system path, add it to the system path
#system path is added to python search. 
# This allows us to import modules from the src 
# folder without having to specify the full path.
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))



## 2. Imports and Shared Settings

Both models use the same dataset, scaler, preprocessing, `input_len`, train/val/test split, and batch format. The shared dataset keeps the full 12-step target window, while each split-horizon model trains on its own 6-step slice.


In [14]:
import json
from datetime import datetime
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader

from basicts.configs import BasicTSForecastingConfig, BasicTSModelConfig
from basicts.launcher import BasicTSLauncher
from basicts.models.iTransformer import iTransformerConfig, iTransformerForForecasting
from basicts.runners.builder import Builder
from basicts.runners.taskflow import BasicTSForecastingTaskFlow
from basicts.scaler import ZScoreScaler
from basicts.utils import BasicTSMode
DATASET_NAME = "ETTh1"
#most papers use 96 input length
INPUT_LEN = 96
#most papers use 96, 192, 336, and 720 input length
FULL_OUTPUT_LEN = 12

SPLIT_OUTPUT_LEN = 6
#etthl has 7 variables
NUM_FEATURES = 7
#Batch sizes like 16, 32, and 64 are normal. 32 is a safe default.
BATCH_SIZE = 32
# common for testing
NUM_EPOCHS = 5
LEARNING_RATE = 1e-3
# Fresh namespace for this notebook so BasicTS does not auto-resume from old/corrupt checkpoints.
RUN_TAG = "mixed_v2"
# this is the shared settings dictionary that both models use
SHARED_CONFIG = {
    "dataset_name": DATASET_NAME,
    "input_len": INPUT_LEN,
    "dataset_params": {
        "input_len": INPUT_LEN,
        "output_len": FULL_OUTPUT_LEN,
        "use_timestamps": False,
        "memmap": False,
    },
    "use_timestamps": False,
    "batch_size": BATCH_SIZE,
    "num_epochs": NUM_EPOCHS,
    "scaler": ZScoreScaler,
    "norm_each_channel": True,
    "rescale": False,
    "metrics": ["MAE", "MSE"],
    "optimizer_params": {"lr": LEARNING_RATE, "weight_decay": 5e-4},
    "gpus": None,
    "train_data_num_workers": 0,
    "val_data_num_workers": 0,
    "test_data_num_workers": 0,
    "save_results": True,
}

# The standalone 12-step models only need BasicTS test_metrics.json.
# Metrics-only evaluation avoids Windows memmap file-lock issues.
FULL_12_STEP_CONFIG = dict(SHARED_CONFIG)
FULL_12_STEP_CONFIG["save_results"] = False


def fresh_checkpoint_dir(model_folder, run_name):
    # Each training launch gets a unique parent folder, so BasicTS cannot resume a stale checkpoint.
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    return str(Path("checkpoints") / RUN_TAG / model_folder / run_name / stamp)


## 3. Shared Shape Check Helper

This helper builds the BasicTS dataset and scaler, runs the same forecasting preprocessing that training uses, and then sends one batch through the selected model.


In [15]:
# for the split forecasitting model, we need to create a custom taskflow that slices the targets and target masks to the desired output length      
#start with the forecasting taskflwo and mofify it 
# BasicTSForecastingTaskFlow is the default data-prep worker.
# It prepares each forecasting batch before the model uses it.
# start with the deault taskflow and then add a change 
class SplitHorizonForecastingTaskFlow(BasicTSForecastingTaskFlow):
    #adding a setting called targest slide whchi is a variable that is used to slice the targets
    # tells the taskflow which targets to keep
    # need self because we need acresss to this specific object
    #creates a variables incide the class object 
    def __init__(self, target_slice):
        self.target_slice = target_slice
    # preprocess 
    def preprocess(self, runner, data):
        #normal work
        data = super().preprocess(runner, data)
        #cuts target values
        data["targets"] = data["targets"][:, self.target_slice, :]
        # tells basicts which target values are valid
        data["targets_mask"] = data["targets_mask"][:, self.target_slice, :]
        return data


def _float_batch(batch):
    return {
        key: value.float() if isinstance(value, torch.Tensor) and value.is_floating_point() else value
        for key, value in batch.items()
    }

# checks to see if the inputs are the right shape the targest are sliced and the prediction matches targer
def preview_shapes(cfg, model_name):
    #build the dataset using basicts
    train_dataset = Builder._build_dataset(cfg, BasicTSMode.TRAIN)
    # puts the dataset into batches gets ready for batches
    train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=False)
    #This grabs the first batch.
    raw_batch = _float_batch(next(iter(train_loader)))
    #creates the scaler and fits it to the training data  
    scaler = Builder._build_scaler(cfg)
    scaler.fit(train_dataset.data)
    # fake runner so the pasicts 
    class PreviewRunner:
        pass
    # Create a fake runner that has cfg and scaler, because taskflow.preprocess expects a runner object.
    runner = PreviewRunner() 
    runner.cfg = cfg
    runner.scaler = scaler
    #prepares the batch normally
    processed_batch = cfg.taskflow.preprocess(runner, dict(raw_batch))
    #build the model from the config and switch it to evaluation mode
    model = cfg.model(cfg.model_config)
    model.eval()
    # o not track gradient as they are only needed for trainig 
    with torch.no_grad():
        #sends processed inputs into the model
        prediction = model(processed_batch["inputs"])
        # then the model outputs future values
        # did the model return a dictionary
        if isinstance(prediction, dict):
            # if it did this extracts onlt the prediction tensor
            prediction = prediction["prediction"]
    #print the shapres to check that the data and model match before the training
    print(f"{model_name} raw inputs shape:       ", tuple(raw_batch["inputs"].shape))
    print(f"{model_name} raw target shape:       ", tuple(raw_batch["targets"].shape))
    print(f"{model_name} processed inputs shape: ", tuple(processed_batch["inputs"].shape))
    print(f"{model_name} target shape:           ", tuple(processed_batch["targets"].shape))
    print(f"{model_name} prediction shape:       ", tuple(prediction.shape))
    #checks
    assert tuple(raw_batch["targets"].shape) == (cfg.batch_size, FULL_OUTPUT_LEN, NUM_FEATURES)
    assert tuple(processed_batch["inputs"].shape) == (cfg.batch_size, INPUT_LEN, NUM_FEATURES)
    assert tuple(processed_batch["targets"].shape) == (cfg.batch_size, SPLIT_OUTPUT_LEN, NUM_FEATURES)
    assert tuple(prediction.shape) == (cfg.batch_size, SPLIT_OUTPUT_LEN, NUM_FEATURES)
    return processed_batch, prediction


## 4. MLP Model

The MLP flattens `[batch_size, input_len, num_features]`, passes the flat vector through linear layers, and reshapes the head output back to `[batch_size, 6, num_features]`. In this split-horizon hybrid, the MLP is responsible for forecast steps 1-6.


In [16]:
#simple mlp forecasting model
#nnmodule is the base class for all pytorch models 
class SimpleMLPForecaster(nn.Module):
    # innit runs when the model is creates 
    def __init__(self, config):
        #calls the parent class setup code
        super().__init__()
        #inportant settings
        self.input_len = config.input_len
        self.output_len = config.output_len
        self.num_features = config.num_features
        flat_input = self.input_len * self.num_features
        flat_output = self.output_len * self.num_features
        #flatten  it into a single vector to the mlp work
        self.net = nn.Sequential(
            nn.Flatten(start_dim=1),
            nn.Linear(flat_input, config.hidden_size),
            #adds nonlinarity
            nn.GELU(),
            #reduce overfitting by randomly setting some values to zero during training
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_size, config.hidden_size),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_size, flat_output),
        )
    #forward pass 
    def forward(self, inputs):
        prediction = self.net(inputs)
        return prediction.view(inputs.size(0), self.output_len, self.num_features)


## 5. MLP Config

This config reuses `BasicTSForecastingConfig` and only changes the model-specific pieces. The shared dataset/scaler/preprocessing settings come from `SHARED_CONFIG`.


In [17]:
#tells basicts hwo to bild and train the MLP
mlp_model_config = BasicTSModelConfig(
    input_len=INPUT_LEN,
    output_len=SPLIT_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    hidden_size=256,
    dropout=0.1,
)

#full basicts training config for MLP
mlp_cfg = BasicTSForecastingConfig(
    model=SimpleMLPForecaster,
    model_config=mlp_model_config,
    taskflow=SplitHorizonForecastingTaskFlow(slice(0, SPLIT_OUTPUT_LEN)),
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/SimpleMLPForecaster/{DATASET_NAME}_{INPUT_LEN}_steps_1_6",
    **SHARED_CONFIG,
)

# Standalone MLP trained to forecast all 12 steps.
mlp_full_model_config = BasicTSModelConfig(
    input_len=INPUT_LEN,
    output_len=FULL_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    hidden_size=256,
    dropout=0.1,
)

mlp_full_cfg = BasicTSForecastingConfig(
    model=SimpleMLPForecaster,
    model_config=mlp_full_model_config,
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/SimpleMLPForecaster/{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    **FULL_12_STEP_CONFIG,
)

mlp_cfg, mlp_full_cfg


(BasicTSForecastingConfig(model=<class '__main__.SimpleMLPForecaster'>, model_config={'input_len': 96, 'output_len': 6, 'num_features': 7, 'hidden_size': 256, 'dropout': 0.1}, dataset_name='ETTh1', taskflow=<__main__.SplitHorizonForecastingTaskFlow object at 0x000001C37914FF50>, callbacks=[], gpus=None, gpu_num=0, seed=42, dataset_type=<class 'basicts.data.tsf_dataset.BasicTSForecastingDataset'>, dataset_params={'input_len': 96, 'output_len': 12, 'use_timestamps': False, 'memmap': False, 'dataset_name': 'ETTh1'}, batch_size=32, null_val=nan, null_to_num=0.0, scaler=<class 'basicts.scaler.z_score_scaler.ZScoreScaler'>, norm_each_channel=True, rescale=False, ddp_find_unused_parameters=False, compile_model=False, metrics=['MAE', 'MSE'], target_metric='MAE', best_metric='min', num_epochs=5, num_steps=None, loss='MAE', optimizer=<class 'torch.optim.adam.Adam'>, optimizer_params={'lr': 0.001, 'weight_decay': 0.0005}, lr=None, lr_scheduler=None, lr_scheduler_params=None, ckpt_save_dir='checkp

## 6. MLP Shape Test

Run this before training. The model prediction and processed target lines must be `(batch_size, 6, num_features)`, while the raw target line remains `(batch_size, 12, num_features)`.


In [18]:
#checker
mlp_batch, mlp_prediction = preview_shapes(mlp_cfg, "MLP")


MLP raw inputs shape:        (32, 96, 7)
MLP raw target shape:        (32, 12, 7)
MLP processed inputs shape:  (32, 96, 7)
MLP target shape:            (32, 6, 7)
MLP prediction shape:        (32, 6, 7)


## 7. Train the MLP

This is the first training run. Leave `RUN_MLP_TRAINING` as `False` while editing or shape-checking, then switch it to `True` when you are ready to train.


In [19]:
#training
RUN_MLP_TRAINING = True
RUN_MLP_12_STEP_TRAINING = True

if RUN_MLP_TRAINING:
    mlp_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "SimpleMLPForecaster",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_1_6",
    )
    print("Training split MLP in:", mlp_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(mlp_cfg)
else:
    print("MLP split-horizon training skipped. Set RUN_MLP_TRAINING = True to train.")

if RUN_MLP_12_STEP_TRAINING:
    mlp_full_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "SimpleMLPForecaster",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    )
    print("Training 12-step MLP in:", mlp_full_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(mlp_full_cfg)
else:
    print("MLP 12-step training skipped. Set RUN_MLP_12_STEP_TRAINING = True to train.")


2026-06-29 17:28:24,944 - BasicTS-launcher - INFO - Launching BasicTS training.
2026-06-29 17:28:24,978 - BasicTS - INFO - Building model.
2026-06-29 17:28:24,982 - BasicTS - INFO - Set ckpt save dir: "checkpoints\mixed_v2\SimpleMLPForecaster\ETTh1_96_steps_1_6\20260629_172824\3da09a93cf705f6bb3c470a8a602c057"
2026-06-29 17:28:24,982 - BasicTS-training - INFO - Initializing training.
2026-06-29 17:28:24,983 - BasicTS-training - INFO - Building train data loader.


Training split MLP in: checkpoints\mixed_v2\SimpleMLPForecaster\ETTh1_96_steps_1_6\20260629_172824


2026-06-29 17:28:26,561 - BasicTS-training - INFO - Set optim: Adam
2026-06-29 17:28:26,562 - BasicTS-training - INFO - Building val data loader.
2026-06-29 17:28:26,573 - BasicTS-training - INFO - Building test data loader.
2026-06-29 17:28:26,579 - BasicTS-training - INFO - Total parameters: 248874
2026-06-29 17:28:26,579 - BasicTS-training - INFO - Trainable parameters: 248874
2026-06-29 17:28:26,584 - BasicTS-training - INFO - Epoch 1 / 5
100%|██████████| 267/267 [00:01<00:00, 184.83it/s]
2026-06-29 17:28:28,031 - BasicTS-training - INFO - Result <train>: [train/time: 1.45 (s), train/loss: 0.3666, train/MAE: 0.3666, train/MSE: 0.2810]
2026-06-29 17:28:28,034 - BasicTS-training - INFO - Start validation.
100%|██████████| 87/87 [00:00<00:00, 680.63it/s]
2026-06-29 17:28:28,164 - BasicTS-training - INFO - Result <val>: [val/time: 0.13 (s), val/loss: 0.4474, val/MAE: 0.4474, val/MSE: 0.4498]
2026-06-29 17:28:28,175 - BasicTS-training - INFO - Checkpoint checkpoints\mixed_v2\SimpleMLPFo

Training 12-step MLP in: checkpoints\mixed_v2\SimpleMLPForecaster\ETTh1_96_steps_1_12\20260629_172835


100%|██████████| 267/267 [00:01<00:00, 177.90it/s]
2026-06-29 17:28:37,191 - BasicTS-training - INFO - Result <train>: [train/time: 1.50 (s), train/loss: 0.3914, train/MAE: 0.3914, train/MSE: 0.3210]
2026-06-29 17:28:37,194 - BasicTS-training - INFO - Start validation.
100%|██████████| 87/87 [00:00<00:00, 648.77it/s]
2026-06-29 17:28:37,330 - BasicTS-training - INFO - Result <val>: [val/time: 0.14 (s), val/loss: 0.4893, val/MAE: 0.4893, val/MSE: 0.5342]
2026-06-29 17:28:37,338 - BasicTS-training - INFO - Checkpoint checkpoints\mixed_v2\SimpleMLPForecaster\ETTh1_96_steps_1_12\20260629_172835\693a107bb0a345185724ce8c8ded5e41\SimpleMLPForecaster_best_val_MAE.pt saved
100%|██████████| 87/87 [00:00<00:00, 656.04it/s]
2026-06-29 17:28:37,472 - BasicTS-training - INFO - Result <test>: [test/time: 0.13 (s), test/loss: 0.5015, test/MAE: 0.5015, test/MSE: 0.4698]
2026-06-29 17:28:37,479 - BasicTS-training - INFO - Checkpoint checkpoints\mixed_v2\SimpleMLPForecaster\ETTh1_96_steps_1_12\20260629_1

## 8. Transformer Model

After the MLP shape check works, use the repo's existing `iTransformerForForecasting`. It receives the same `[batch_size, input_len, num_features]` input and returns `[batch_size, 6, num_features]`. In this split-horizon hybrid, the Transformer is responsible for forecast steps 7-12.


In [20]:
#transfoemr model build it
transformer_model_config = iTransformerConfig(
    input_len=INPUT_LEN,
    output_len=SPLIT_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    hidden_size=64,
    intermediate_size=128,
    n_heads=4,
    num_layers=2,
    dropout=0.1,
    use_revin=True,
)

transformer_cfg = BasicTSForecastingConfig(
    model=iTransformerForForecasting,
    model_config=transformer_model_config,
    taskflow=SplitHorizonForecastingTaskFlow(slice(SPLIT_OUTPUT_LEN, FULL_OUTPUT_LEN)),
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/iTransformerForForecasting/{DATASET_NAME}_{INPUT_LEN}_steps_7_12",
    **SHARED_CONFIG,
)

# Standalone Transformer trained to forecast all 12 steps.
transformer_full_model_config = iTransformerConfig(
    input_len=INPUT_LEN,
    output_len=FULL_OUTPUT_LEN,
    num_features=NUM_FEATURES,
    hidden_size=64,
    intermediate_size=128,
    n_heads=4,
    num_layers=2,
    dropout=0.1,
    use_revin=True,
)

transformer_full_cfg = BasicTSForecastingConfig(
    model=iTransformerForForecasting,
    model_config=transformer_full_model_config,
    ckpt_save_dir=f"checkpoints/{RUN_TAG}/iTransformerForForecasting/{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    **FULL_12_STEP_CONFIG,
)

transformer_cfg, transformer_full_cfg


(BasicTSForecastingConfig(model=<class 'basicts.models.iTransformer.arch.itransformer_arch.iTransformerForForecasting'>, model_config=iTransformerConfig(input_len=96, output_len=6, num_features=7, num_classes=None, hidden_size=64, n_heads=4, intermediate_size=128, hidden_act='gelu', num_layers=2, dropout=0.1, use_revin=True, output_attentions=False), dataset_name='ETTh1', taskflow=<__main__.SplitHorizonForecastingTaskFlow object at 0x000001C37917E150>, callbacks=[], gpus=None, gpu_num=0, seed=42, dataset_type=<class 'basicts.data.tsf_dataset.BasicTSForecastingDataset'>, dataset_params={'input_len': 96, 'output_len': 12, 'use_timestamps': False, 'memmap': False, 'dataset_name': 'ETTh1'}, batch_size=32, null_val=nan, null_to_num=0.0, scaler=<class 'basicts.scaler.z_score_scaler.ZScoreScaler'>, norm_each_channel=True, rescale=False, ddp_find_unused_parameters=False, compile_model=False, metrics=['MAE', 'MSE'], target_metric='MAE', best_metric='min', num_epochs=5, num_steps=None, loss='MAE

## 9. Transformer Shape Test

Run this after the MLP section works. It uses the same shared data pipeline and checks that the Transformer predicts only its 6-step split-horizon target.


In [21]:
#check the shapes
transformer_batch, transformer_prediction = preview_shapes(transformer_cfg, "Transformer")


Transformer raw inputs shape:        (32, 96, 7)
Transformer raw target shape:        (32, 12, 7)
Transformer processed inputs shape:  (32, 96, 7)
Transformer target shape:            (32, 6, 7)
Transformer prediction shape:        (32, 6, 7)


## 10. Train the Transformer

Use the same dataset, scaler, preprocessing, and `input_len` as the MLP. The shared dataset still contains the full 12-step target window, but the Transformer taskflow slices that target to steps 7-12.


In [22]:
#train trasnfoemr
RUN_TRANSFORMER_TRAINING = True
RUN_TRANSFORMER_12_STEP_TRAINING = True

if RUN_TRANSFORMER_TRAINING:
    transformer_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "iTransformerForForecasting",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_7_12",
    )
    print("Training split Transformer in:", transformer_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(transformer_cfg)
else:
    print("Transformer split-horizon training skipped. Set RUN_TRANSFORMER_TRAINING = True after the MLP works.")

if RUN_TRANSFORMER_12_STEP_TRAINING:
    transformer_full_cfg.ckpt_save_dir = fresh_checkpoint_dir(
        "iTransformerForForecasting",
        f"{DATASET_NAME}_{INPUT_LEN}_steps_1_12",
    )
    print("Training 12-step Transformer in:", transformer_full_cfg.ckpt_save_dir)
    BasicTSLauncher.launch_training(transformer_full_cfg)
else:
    print("Transformer 12-step training skipped. Set RUN_TRANSFORMER_12_STEP_TRAINING = True to train.")


2026-06-29 17:28:44,918 - BasicTS-launcher - INFO - Launching BasicTS training.
2026-06-29 17:28:44,922 - BasicTS - INFO - Building model.
2026-06-29 17:28:44,926 - BasicTS - INFO - Set ckpt save dir: "checkpoints\mixed_v2\iTransformerForForecasting\ETTh1_96_steps_7_12\20260629_172844\da5a225660a3a9dd77f3deffc31eefbb"
2026-06-29 17:28:44,927 - BasicTS-training - INFO - Initializing training.
2026-06-29 17:28:44,928 - BasicTS-training - INFO - Building train data loader.
2026-06-29 17:28:44,932 - BasicTS-training - INFO - Set optim: Adam
2026-06-29 17:28:44,932 - BasicTS-training - INFO - Building val data loader.
2026-06-29 17:28:44,934 - BasicTS-training - INFO - Building test data loader.
2026-06-29 17:28:44,935 - BasicTS-training - INFO - Total parameters: 73670
2026-06-29 17:28:44,936 - BasicTS-training - INFO - Trainable parameters: 73670
2026-06-29 17:28:44,936 - BasicTS-training - INFO - Epoch 1 / 5


Training split Transformer in: checkpoints\mixed_v2\iTransformerForForecasting\ETTh1_96_steps_7_12\20260629_172844


100%|██████████| 267/267 [00:04<00:00, 60.51it/s]
2026-06-29 17:28:49,352 - BasicTS-training - INFO - Result <train>: [train/time: 4.41 (s), train/loss: 0.3740, train/MAE: 0.3740, train/MSE: 0.3016]
2026-06-29 17:28:49,354 - BasicTS-training - INFO - Start validation.
100%|██████████| 87/87 [00:00<00:00, 174.49it/s]
2026-06-29 17:28:49,854 - BasicTS-training - INFO - Result <val>: [val/time: 0.50 (s), val/loss: 0.4233, val/MAE: 0.4233, val/MSE: 0.4048]
2026-06-29 17:28:49,860 - BasicTS-training - INFO - Checkpoint checkpoints\mixed_v2\iTransformerForForecasting\ETTh1_96_steps_7_12\20260629_172844\da5a225660a3a9dd77f3deffc31eefbb\iTransformerForForecasting_best_val_MAE.pt saved
100%|██████████| 87/87 [00:00<00:00, 180.96it/s]
2026-06-29 17:28:50,344 - BasicTS-training - INFO - Result <test>: [test/time: 0.48 (s), test/loss: 0.3770, test/MAE: 0.3770, test/MSE: 0.3465]
2026-06-29 17:28:50,354 - BasicTS-training - INFO - Checkpoint checkpoints\mixed_v2\iTransformerForForecasting\ETTh1_96_s

Training 12-step Transformer in: checkpoints\mixed_v2\iTransformerForForecasting\ETTh1_96_steps_1_12\20260629_172912


100%|██████████| 267/267 [00:05<00:00, 47.60it/s]
2026-06-29 17:29:18,059 - BasicTS-training - INFO - Result <train>: [train/time: 5.61 (s), train/loss: 0.3500, train/MAE: 0.3500, train/MSE: 0.2658]
2026-06-29 17:29:18,059 - BasicTS-training - INFO - Start validation.
100%|██████████| 87/87 [00:00<00:00, 146.15it/s]
2026-06-29 17:29:18,659 - BasicTS-training - INFO - Result <val>: [val/time: 0.60 (s), val/loss: 0.3788, val/MAE: 0.3788, val/MSE: 0.3323]
2026-06-29 17:29:18,667 - BasicTS-training - INFO - Checkpoint checkpoints\mixed_v2\iTransformerForForecasting\ETTh1_96_steps_1_12\20260629_172912\8829c0bc969793aabf5cbfc3ca00f94e\iTransformerForForecasting_best_val_MAE.pt saved
100%|██████████| 87/87 [00:00<00:00, 138.07it/s]
2026-06-29 17:29:19,304 - BasicTS-training - INFO - Result <test>: [test/time: 0.64 (s), test/loss: 0.3403, test/MAE: 0.3403, test/MSE: 0.2893]
2026-06-29 17:29:19,316 - BasicTS-training - INFO - Checkpoint checkpoints\mixed_v2\iTransformerForForecasting\ETTh1_96_s

## 11. Hybrid Prediction: MLP Steps 1-6, Transformer Steps 7-12

This is a true split-horizon hybrid. The MLP predicts only the first 6 forecast steps, the Transformer predicts only the next 6 forecast steps from the same input window, and the final hybrid prediction is the time-axis concatenation of those two 6-step outputs.


In [23]:
import numpy as np

#checks that the mlp and transformer both used the same batch
assert torch.equal(mlp_batch["inputs"], transformer_batch["inputs"])

# Split-horizon hybrid: MLP predicts steps 1-6, Transformer predicts steps 7-12.
mlp_pred = mlp_prediction
transformer_pred = transformer_prediction
#this combines the two step prediction into one 12 step hybrui prediction
hybrid_pred = torch.cat([mlp_pred, transformer_pred], dim=1)
#combines the targests together
hybrid_targets = torch.cat([mlp_batch["targets"], transformer_batch["targets"]], dim=1)
# checks 
print("MLP prediction shape:", tuple(mlp_pred.shape))
print("Transformer prediction shape:", tuple(transformer_pred.shape))
print("Hybrid prediction shape:", tuple(hybrid_pred.shape))
print("Target shape:", tuple(hybrid_targets.shape))

assert tuple(mlp_pred.shape) == (BATCH_SIZE, SPLIT_OUTPUT_LEN, NUM_FEATURES)
assert tuple(transformer_pred.shape) == (BATCH_SIZE, SPLIT_OUTPUT_LEN, NUM_FEATURES)
assert tuple(hybrid_pred.shape) == (BATCH_SIZE, FULL_OUTPUT_LEN, NUM_FEATURES)
assert tuple(hybrid_targets.shape) == (BATCH_SIZE, FULL_OUTPUT_LEN, NUM_FEATURES)

#find the last saved prediction
def latest_prediction_file(cfg):
    prediction_files = sorted(
        Path(cfg.ckpt_save_dir).glob("*/test_results/prediction.npy"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not prediction_files:
        raise FileNotFoundError(
            f"No prediction.npy found under {cfg.ckpt_save_dir}. Train/evaluate this model with save_results=True first."
        )
    return prediction_files[0]

#load the prediction arrays
def load_basicts_array(path, shape):
    # BasicTS writes these files as raw memmaps, even though the file names end in .npy.
    array = np.memmap(path, dtype=np.float32, mode="r", shape=shape)
    return np.asarray(array)

#loads the largets
def load_basicts_prediction_and_targets(cfg, output_len):
    prediction_path = latest_prediction_file(cfg)
    targets_path = prediction_path.parent / "targets.npy"
    if not targets_path.exists():
        raise FileNotFoundError(f"No targets.npy found next to {prediction_path}")

    test_dataset = Builder._build_dataset(cfg, BasicTSMode.TEST)
    shape = (len(test_dataset), output_len, NUM_FEATURES)
    prediction = load_basicts_array(prediction_path, shape)
    targets = load_basicts_array(targets_path, shape)
    return prediction, targets


mlp_test_pred, mlp_test_targets = load_basicts_prediction_and_targets(mlp_cfg, SPLIT_OUTPUT_LEN)
transformer_test_pred, transformer_test_targets = load_basicts_prediction_and_targets(transformer_cfg, SPLIT_OUTPUT_LEN)
#bombines the test predictions
hybrid_test_pred = np.concatenate([mlp_test_pred, transformer_test_pred], axis=1)
hybrid_test_targets = np.concatenate([mlp_test_targets, transformer_test_targets], axis=1)

assert mlp_test_pred.shape == mlp_test_targets.shape
assert transformer_test_pred.shape == transformer_test_targets.shape
assert mlp_test_pred.shape == transformer_test_pred.shape
assert hybrid_test_pred.shape == hybrid_test_targets.shape
assert hybrid_test_pred.shape[1] == FULL_OUTPUT_LEN

hybrid_save_path = Path("checkpoints/hybrid_split_horizon_mlp_steps_1_6_transformer_steps_7_12_ETTh1_96_12_prediction.npy")
np.save(hybrid_save_path, hybrid_test_pred)


MLP prediction shape: (32, 6, 7)
Transformer prediction shape: (32, 6, 7)
Hybrid prediction shape: (32, 12, 7)
Target shape: (32, 12, 7)


## 12. MAE/MSE Comparison

This cell computes MAE/MSE directly from the saved predictions and targets. The MLP is compared only against target steps 1-6, the Transformer only against target steps 7-12, and the hybrid against the full 12-step target.


In [24]:
#use math to compute metrics
def compute_metrics(prediction, targets):
    return {
        "MAE": float(np.mean(np.abs(prediction - targets))),
        "MSE": float(np.mean((prediction - targets) ** 2)),
    }

def latest_metrics_file(cfg):
    metrics_files = sorted(
        Path(cfg.ckpt_save_dir).glob("*/test_metrics.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    if not metrics_files:
        raise FileNotFoundError(f"No test_metrics.json found under {cfg.ckpt_save_dir}. Train this model first.")
    return metrics_files[0]


def load_test_metrics(cfg):
    metrics_path = latest_metrics_file(cfg)
    with metrics_path.open("r", encoding="utf-8") as f:
        metrics = json.load(f)
    return metrics.get("overall", metrics)


# Compare the split models, hybrid, and standalone 12-step models.
comparison = {
    "MLP split steps 1-6": compute_metrics(mlp_test_pred, mlp_test_targets),
    "Transformer split steps 7-12": compute_metrics(transformer_test_pred, transformer_test_targets),
    "Hybrid split steps 1-12": compute_metrics(hybrid_test_pred, hybrid_test_targets),
    "MLP full steps 1-12": load_test_metrics(mlp_full_cfg),
    "Transformer full steps 1-12": load_test_metrics(transformer_full_cfg),
}

print("MLP target shape:", mlp_test_targets.shape)
print("Transformer target shape:", transformer_test_targets.shape)
print("Hybrid target shape:", hybrid_test_targets.shape)
print("MLP full 12-step metrics file:", latest_metrics_file(mlp_full_cfg))
print("Transformer full 12-step metrics file:", latest_metrics_file(transformer_full_cfg))

for model_name, metrics in comparison.items():
    print(model_name)
    for metric_name in ["MAE", "MSE"]:
        print(f"  {metric_name}: {metrics[metric_name]:.6f}")


MLP target shape: (2773, 6, 7)
Transformer target shape: (2773, 6, 7)
Hybrid target shape: (2773, 12, 7)
MLP full 12-step metrics file: checkpoints\mixed_v2\SimpleMLPForecaster\ETTh1_96_steps_1_12\20260629_172835\693a107bb0a345185724ce8c8ded5e41\test_metrics.json
Transformer full 12-step metrics file: checkpoints\mixed_v2\iTransformerForForecasting\ETTh1_96_steps_1_12\20260629_172912\8829c0bc969793aabf5cbfc3ca00f94e\test_metrics.json
MLP split steps 1-6
  MAE: 0.445109
  MSE: 0.360804
Transformer split steps 7-12
  MAE: 0.358287
  MSE: 0.323265
Hybrid split steps 1-12
  MAE: 0.401698
  MSE: 0.342034
MLP full steps 1-12
  MAE: 0.472694
  MSE: 0.423989
Transformer full steps 1-12
  MAE: 0.332324
  MSE: 0.277150


## 13. Efficiency Comparison

This cell compares model size and average batch prediction time. The MLP timing is for its 6-step prediction, the Transformer timing is for its 6-step prediction, and the hybrid timing is the sum of running both 6-step models once.


In [25]:

import time

#counts model size
def count_trainable_parameters(model):
    return sum(param.numel() for param in model.parameters() if param.requires_grad)

#takes one batch and runs the model many times
def time_model_prediction(model, batch, expected_output_len, repeats=50, warmup=5):
    #measure how fast a model makes predictions in one batch
    # check if its on cpu or gpu
    device = next(model.parameters()).device
    # moves input to the same device
    inputs = batch["inputs"].to(device)
    model.eval()

    with torch.no_grad():
        for _ in range(warmup):
            _ = model(inputs)
    #predicts the batch 50 times
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(repeats):
            prediction = model(inputs)
    end = time.perf_counter()

    if isinstance(prediction, dict):
        prediction = prediction["prediction"]
    assert tuple(prediction.shape) == (inputs.size(0), expected_output_len, NUM_FEATURES)
    return (end - start) / repeats


mlp_efficiency_model = mlp_cfg.model(mlp_cfg.model_config)
transformer_efficiency_model = transformer_cfg.model(transformer_cfg.model_config)
mlp_full_efficiency_model = mlp_full_cfg.model(mlp_full_cfg.model_config)
transformer_full_efficiency_model = transformer_full_cfg.model(transformer_full_cfg.model_config)

mlp_params = count_trainable_parameters(mlp_efficiency_model)
transformer_params = count_trainable_parameters(transformer_efficiency_model)
mlp_full_params = count_trainable_parameters(mlp_full_efficiency_model)
transformer_full_params = count_trainable_parameters(transformer_full_efficiency_model)
hybrid_params = mlp_params + transformer_params

mlp_time = time_model_prediction(mlp_efficiency_model, mlp_batch, SPLIT_OUTPUT_LEN)
transformer_time = time_model_prediction(transformer_efficiency_model, transformer_batch, SPLIT_OUTPUT_LEN)
mlp_full_time = time_model_prediction(mlp_full_efficiency_model, mlp_batch, FULL_OUTPUT_LEN)
transformer_full_time = time_model_prediction(transformer_full_efficiency_model, transformer_batch, FULL_OUTPUT_LEN)
hybrid_time = mlp_time + transformer_time

print("MLP parameters:", mlp_params)
print("Transformer parameters:", transformer_params)
print("Hybrid parameters:", hybrid_params)
print("MLP full 12-step parameters:", mlp_full_params)
print("Transformer full 12-step parameters:", transformer_full_params)

print(f"MLP 6-step avg prediction time: {mlp_time:.6f} seconds")
print(f"Transformer 6-step avg prediction time: {transformer_time:.6f} seconds")
print(f"Hybrid avg prediction time: {hybrid_time:.6f} seconds")
print(f"MLP full 12-step avg prediction time: {mlp_full_time:.6f} seconds")
print(f"Transformer full 12-step avg prediction time: {transformer_full_time:.6f} seconds")

rows = [
    ("MLP split 1-6", comparison["MLP split steps 1-6"], mlp_params, mlp_time),
    ("Transformer split 7-12", comparison["Transformer split steps 7-12"], transformer_params, transformer_time),
    ("Hybrid split 1-12", comparison["Hybrid split steps 1-12"], hybrid_params, hybrid_time),
    ("MLP full 1-12", comparison["MLP full steps 1-12"], mlp_full_params, mlp_full_time),
    ("Transformer full 1-12", comparison["Transformer full steps 1-12"], transformer_full_params, transformer_full_time),
]

print(f"{'Model':<14} {'MAE':>10} {'MSE':>10} {'Params':>12} {'Avg batch sec':>15}")
print("-" * 65)
for model_name, metrics, params, avg_time in rows:
    print(f"{model_name:<14} {metrics['MAE']:>10.6f} {metrics['MSE']:>10.6f} {params:>12,} {avg_time:>15.6f}")


MLP parameters: 248874
Transformer parameters: 73670
Hybrid parameters: 322544
MLP full 12-step parameters: 259668
Transformer full 12-step parameters: 74060
MLP 6-step avg prediction time: 0.000698 seconds
Transformer 6-step avg prediction time: 0.004860 seconds
Hybrid avg prediction time: 0.005558 seconds
MLP full 12-step avg prediction time: 0.000820 seconds
Transformer full 12-step avg prediction time: 0.005650 seconds
Model                 MAE        MSE       Params   Avg batch sec
-----------------------------------------------------------------
MLP split 1-6    0.445109   0.360804      248,874        0.000698
Transformer split 7-12   0.358287   0.323265       73,670        0.004860
Hybrid split 1-12   0.401698   0.342034      322,544        0.005558
MLP full 1-12    0.472694   0.423989      259,668        0.000820
Transformer full 1-12   0.332324   0.277150       74,060        0.005650


## 14. Selector-Based Hybrid: Hard and Soft Weighted Selection

In this section, we move beyond the fixed split-horizon hybrid (MLP steps 1-6, Transformer steps 7-12) to intelligent selection mechanisms:

1. **Hard Selector**: For each forecast step, choose the model with the lowest MAE on that step.
2. **Soft Weighted Selector**: For each step, blend predictions using weights derived from inverse MAE (lower error = higher weight).

This approach allows the hybrid to leverage whichever model performs best at each forecast horizon.


In [26]:
# Load full 12-step model predictions for selector comparison.
# These full models forecast all 12 steps, allowing step-by-step model selection.

print("Loading full 12-step model predictions...")

mlp_full_test_pred, mlp_full_test_targets = load_basicts_prediction_and_targets(mlp_full_cfg, FULL_OUTPUT_LEN)
transformer_full_test_pred, transformer_full_test_targets = load_basicts_prediction_and_targets(transformer_full_cfg, FULL_OUTPUT_LEN)

print(f"MLP full pred shape: {mlp_full_test_pred.shape}")
print(f"MLP full targets shape: {mlp_full_test_targets.shape}")
print(f"Transformer full pred shape: {transformer_full_test_pred.shape}")
print(f"Transformer full targets shape: {transformer_full_test_targets.shape}")

# Verify all shapes are consistent
assert mlp_full_test_pred.shape == (mlp_full_test_pred.shape[0], FULL_OUTPUT_LEN, NUM_FEATURES)
assert transformer_full_test_pred.shape == (transformer_full_test_pred.shape[0], FULL_OUTPUT_LEN, NUM_FEATURES)
assert mlp_full_test_pred.shape == mlp_full_test_targets.shape
assert transformer_full_test_pred.shape == transformer_full_test_targets.shape


Loading full 12-step model predictions...
MLP full pred shape: (2773, 12, 7)
MLP full targets shape: (2773, 12, 7)
Transformer full pred shape: (2773, 12, 7)
Transformer full targets shape: (2773, 12, 7)


In [27]:
# Hard Selector: For each forecast step and feature, choose the model with the lowest MAE.

print("\n=== HARD SELECTOR (Step-wise Best Model) ===")

# Compute per-step MAE for each model
# Predictions shape: (num_samples, 12, num_features)
# After mean over samples: (12, num_features)

mlp_full_per_step_mae = np.mean(
    np.abs(mlp_full_test_pred - mlp_full_test_targets),
    axis=0
)

transformer_full_per_step_mae = np.mean(
    np.abs(transformer_full_test_pred - transformer_full_test_targets),
    axis=0
)

print(f"MLP full per-step MAE shape: {mlp_full_per_step_mae.shape}")
print(f"Transformer full per-step MAE shape: {transformer_full_per_step_mae.shape}")

# For each step and feature, select the model with lower MAE
# hard_selector_mask[step, feature] = 0 for MLP, 1 for Transformer
hard_selector_mask = (
    transformer_full_per_step_mae < mlp_full_per_step_mae
).astype(int)

print(f"Hard selector mask shape: {hard_selector_mask.shape}")

# Build hard-selected hybrid prediction by picking per-step/feature best model
# hard_selector_mask shape: (12, 7)
# hard_selector_mask[np.newaxis, :, :] shape: (1, 12, 7)
# prediction shape: (num_samples, 12, 7)
hard_selected_pred = np.where(
    hard_selector_mask[np.newaxis, :, :],
    transformer_full_test_pred,
    mlp_full_test_pred,
)

# For targets, use the full 12-step target
hard_selected_targets = mlp_full_test_targets

print(f"Hard-selected prediction shape: {hard_selected_pred.shape}")
print(f"Hard-selected targets shape: {hard_selected_targets.shape}")

assert hard_selected_pred.shape == (hard_selected_pred.shape[0], FULL_OUTPUT_LEN, NUM_FEATURES)
assert hard_selected_targets.shape == (hard_selected_targets.shape[0], FULL_OUTPUT_LEN, NUM_FEATURES)
assert hard_selected_pred.shape == hard_selected_targets.shape

# Count how often Transformer was selected per step
print("\nHard selector: Model choices per step (% Transformer):")
for step in range(FULL_OUTPUT_LEN):
    pct_transformer = np.mean(hard_selector_mask[step, :]) * 100
    print(f"  Step {step+1}: {pct_transformer:.1f}% Transformer, {100-pct_transformer:.1f}% MLP")


=== HARD SELECTOR (Step-wise Best Model) ===
MLP full per-step MAE shape: (12, 7)
Transformer full per-step MAE shape: (12, 7)
Hard selector mask shape: (12, 7)
Hard-selected prediction shape: (2773, 12, 7)
Hard-selected targets shape: (2773, 12, 7)

Hard selector: Model choices per step (% Transformer):
  Step 1: 100.0% Transformer, 0.0% MLP
  Step 2: 100.0% Transformer, 0.0% MLP
  Step 3: 100.0% Transformer, 0.0% MLP
  Step 4: 85.7% Transformer, 14.3% MLP
  Step 5: 85.7% Transformer, 14.3% MLP
  Step 6: 85.7% Transformer, 14.3% MLP
  Step 7: 100.0% Transformer, 0.0% MLP
  Step 8: 100.0% Transformer, 0.0% MLP
  Step 9: 100.0% Transformer, 0.0% MLP
  Step 10: 100.0% Transformer, 0.0% MLP
  Step 11: 100.0% Transformer, 0.0% MLP
  Step 12: 100.0% Transformer, 0.0% MLP


In [28]:
# Soft Weighted Selector: Blend predictions using weights derived from inverse MAE.
# Lower MAE gets higher weight; predictions are then weighted-averaged.

print("\n=== SOFT WEIGHTED SELECTOR (Step-wise Weighted Average) ===")

# Compute MAE for each forecast step and feature
# Prediction shape: (num_samples, 12, 7)
# After mean over samples: (12, 7)
mlp_full_per_step_mae_values = np.mean(
    np.abs(mlp_full_test_pred - mlp_full_test_targets),
    axis=0
)

transformer_full_per_step_mae_values = np.mean(
    np.abs(transformer_full_test_pred - transformer_full_test_targets),
    axis=0
)

print(f"MLP full per-step MAE shape: {mlp_full_per_step_mae_values.shape}")
print(f"Transformer full per-step MAE shape: {transformer_full_per_step_mae_values.shape}")

# Compute inverse weights
epsilon = 1e-6

mlp_weights = 1.0 / (mlp_full_per_step_mae_values + epsilon)
transformer_weights = 1.0 / (transformer_full_per_step_mae_values + epsilon)

# Normalize weights so MLP weight + Transformer weight = 1
total_weights = mlp_weights + transformer_weights

mlp_weights_normalized = mlp_weights / total_weights
transformer_weights_normalized = transformer_weights / total_weights

print(f"MLP normalized weights shape: {mlp_weights_normalized.shape}")
print(f"Transformer normalized weights shape: {transformer_weights_normalized.shape}")

# Build weighted prediction
# weights shape: (12, 7)
# weights[np.newaxis, :, :] shape: (1, 12, 7)
# predictions shape: (num_samples, 12, 7)
weighted_pred = (
    mlp_weights_normalized[np.newaxis, :, :] * mlp_full_test_pred
    + transformer_weights_normalized[np.newaxis, :, :] * transformer_full_test_pred
)

weighted_targets = mlp_full_test_targets

print(f"Weighted prediction shape: {weighted_pred.shape}")
print(f"Weighted targets shape: {weighted_targets.shape}")

assert weighted_pred.shape == (weighted_pred.shape[0], FULL_OUTPUT_LEN, NUM_FEATURES)
assert weighted_targets.shape == (weighted_targets.shape[0], FULL_OUTPUT_LEN, NUM_FEATURES)
assert weighted_pred.shape == weighted_targets.shape

# Show average weight per step across all 7 features
print("\nSoft weights per step, averaged across features:")
print(f"{'Step':<6} {'MLP Weight':<12} {'Transformer Weight':<18}")
print("-" * 36)

for step in range(FULL_OUTPUT_LEN):
    mlp_step_weight = np.mean(mlp_weights_normalized[step, :])
    transformer_step_weight = np.mean(transformer_weights_normalized[step, :])

    print(f"{step+1:<6} {mlp_step_weight:<12.4f} {transformer_step_weight:<18.4f}")


=== SOFT WEIGHTED SELECTOR (Step-wise Weighted Average) ===
MLP full per-step MAE shape: (12, 7)
Transformer full per-step MAE shape: (12, 7)
MLP normalized weights shape: (12, 7)
Transformer normalized weights shape: (12, 7)
Weighted prediction shape: (2773, 12, 7)
Weighted targets shape: (2773, 12, 7)

Soft weights per step, averaged across features:
Step   MLP Weight   Transformer Weight
------------------------------------
1      0.3843       0.6157            
2      0.4067       0.5933            
3      0.4182       0.5818            
4      0.4200       0.5800            
5      0.4055       0.5945            
6      0.4090       0.5910            
7      0.3950       0.6050            
8      0.3933       0.6067            
9      0.3951       0.6049            
10     0.3902       0.6098            
11     0.3941       0.6059            
12     0.3905       0.6095            


In [29]:
# Comprehensive metrics table with all 7 models (including new selectors)

print("\n=== COMPREHENSIVE METRICS COMPARISON (All 7 Models) ===\n")

# Build comparison dict with all models
all_comparison = {
    "MLP split steps 1-6": compute_metrics(mlp_test_pred, mlp_test_targets),
    "Transformer split steps 7-12": compute_metrics(transformer_test_pred, transformer_test_targets),
    "Fixed split hybrid 1-12": compute_metrics(hybrid_test_pred, hybrid_test_targets),
    "MLP full steps 1-12": compute_metrics(mlp_full_test_pred, mlp_full_test_targets),
    "Transformer full steps 1-12": compute_metrics(transformer_full_test_pred, transformer_full_test_targets),
    "Hard selector hybrid 1-12": compute_metrics(hard_selected_pred, hard_selected_targets),
    "Soft weighted hybrid 1-12": compute_metrics(weighted_pred, weighted_targets),
}

# Print table
print(f"{'Model':<35} {'MAE':>12} {'MSE':>12}")
print("-" * 60)
for model_name in [
    "MLP split steps 1-6",
    "Transformer split steps 7-12",
    "Fixed split hybrid 1-12",
    "MLP full steps 1-12",
    "Transformer full steps 1-12",
    "Hard selector hybrid 1-12",
    "Soft weighted hybrid 1-12",
]:
    metrics = all_comparison[model_name]
    mae = metrics["MAE"]
    mse = metrics["MSE"]
    print(f"{model_name:<35} {mae:>12.6f} {mse:>12.6f}")

# Identify best performers
best_mae_model = min(all_comparison.items(), key=lambda x: x[1]["MAE"])
best_mse_model = min(all_comparison.items(), key=lambda x: x[1]["MSE"])

print("\n" + "=" * 60)
print(f"Best MAE: {best_mae_model[0]:<30} {best_mae_model[1]['MAE']:.6f}")
print(f"Best MSE: {best_mse_model[0]:<30} {best_mse_model[1]['MSE']:.6f}")



=== COMPREHENSIVE METRICS COMPARISON (All 7 Models) ===

Model                                        MAE          MSE
------------------------------------------------------------
MLP split steps 1-6                     0.445109     0.360804
Transformer split steps 7-12            0.358287     0.323265
Fixed split hybrid 1-12                 0.401698     0.342034
MLP full steps 1-12                     0.467399     0.418559
Transformer full steps 1-12             0.329430     0.274991
Hard selector hybrid 1-12               0.329022     0.275092
Soft weighted hybrid 1-12               0.337959     0.280933

Best MAE: Hard selector hybrid 1-12      0.329022
Best MSE: Transformer full steps 1-12    0.274991


In [30]:
# Save selector hybrid predictions to checkpoints/ for future use

print("\n=== SAVING SELECTOR PREDICTIONS ===\n")

# Hard selector predictions
hard_selector_save_path = Path("checkpoints/hard_selector_hybrid_mlp_transformer_ETTh1_96_12_prediction.npy")
np.save(hard_selector_save_path, hard_selected_pred)
print(f"Saved hard selector predictions: {hard_selector_save_path}")
print(f"  Shape: {hard_selected_pred.shape}")

# Soft weighted selector predictions
weighted_selector_save_path = Path("checkpoints/soft_weighted_hybrid_mlp_transformer_ETTh1_96_12_prediction.npy")
np.save(weighted_selector_save_path, weighted_pred)
print(f"Saved soft weighted predictions: {weighted_selector_save_path}")
print(f"  Shape: {weighted_pred.shape}")

# Also save the weights and hard selector mask for reference
weights_info = {
    "mlp_weights": mlp_weights_normalized.tolist(),
    "transformer_weights": transformer_weights_normalized.tolist(),
    "hard_selector_mask": hard_selector_mask.tolist(),
    "description": "Hard selector mask: 0=MLP, 1=Transformer. Soft weights: normalized inverse MAE per step.",
}
weights_save_path = Path("checkpoints/selector_metadata_ETTh1_96_12.json")
with weights_save_path.open("w", encoding="utf-8") as f:
    json.dump(weights_info, f, indent=2)
print(f"Saved selector metadata: {weights_save_path}")

print("\n✓ All selector predictions and metadata saved successfully.")



=== SAVING SELECTOR PREDICTIONS ===

Saved hard selector predictions: checkpoints\hard_selector_hybrid_mlp_transformer_ETTh1_96_12_prediction.npy
  Shape: (2773, 12, 7)
Saved soft weighted predictions: checkpoints\soft_weighted_hybrid_mlp_transformer_ETTh1_96_12_prediction.npy
  Shape: (2773, 12, 7)
Saved selector metadata: checkpoints\selector_metadata_ETTh1_96_12.json

✓ All selector predictions and metadata saved successfully.


In [32]:
# Summary and interpretation of selector approaches

print("\n" + "="*70)
print("SELECTOR-BASED HYBRID SUMMARY")
print("="*70)

print("""
Three hybrid approaches have been compared:

1. FIXED SPLIT HYBRID
   - MLP forecasts steps 1-6
   - Transformer forecasts steps 7-12
   - Simple concatenation; uses pre-trained split-horizon models
   - Baseline: steps are split by design, not by performance

2. HARD SELECTOR HYBRID
   - For each forecast step, choose the single best model (lowest MAE)
   - Step-by-step selection: some steps use MLP, others use Transformer
   - Binary choice per step: no blending
   - Advantage: leverages each model's strength at its best steps
   - File: hard_selector_hybrid_mlp_transformer_ETTh1_96_12_prediction.npy

3. SOFT WEIGHTED HYBRID
   - For each forecast step, weight-average both model predictions
   - Weights based on inverse MAE: lower error → higher weight
   - Smooth blending: captures model complementarity
   - Advantage: reduces impact of single model failures
   - File: soft_weighted_hybrid_mlp_transformer_ETTh1_96_12_prediction.npy

All selector hybrids use FULL 12-STEP model predictions (mlp_full_cfg and 
transformer_full_cfg), enabling true per-step selection. The fixed split 
hybrid combines pre-trained 6-step models (mlp_cfg and transformer_cfg).

NEXT STEPS:
- Run this notebook to compute predictions after training full models
- Compare metrics: MAE and MSE should improve with intelligent selection
- Analyze which steps each model excels at (see hard selector mask printout)
- Consider the weight distribution to understand model complementarity
""")

print("="*70)
print("✓ Selector-based hybrid implementation complete.")
print("="*70)



SELECTOR-BASED HYBRID SUMMARY

Three hybrid approaches have been compared:

1. FIXED SPLIT HYBRID
   - MLP forecasts steps 1-6
   - Transformer forecasts steps 7-12
   - Simple concatenation; uses pre-trained split-horizon models
   - Baseline: steps are split by design, not by performance

2. HARD SELECTOR HYBRID
   - For each forecast step, choose the single best model (lowest MAE)
   - Step-by-step selection: some steps use MLP, others use Transformer
   - Binary choice per step: no blending
   - Advantage: leverages each model's strength at its best steps
   - File: hard_selector_hybrid_mlp_transformer_ETTh1_96_12_prediction.npy

3. SOFT WEIGHTED HYBRID
   - For each forecast step, weight-average both model predictions
   - Weights based on inverse MAE: lower error → higher weight
   - Smooth blending: captures model complementarity
   - Advantage: reduces impact of single model failures
   - File: soft_weighted_hybrid_mlp_transformer_ETTh1_96_12_prediction.npy

All selector hybri